# Text-I-JEPA: предсказание скрытых текстовых блоков в latent space

**Цель:** воспроизвести подход I-JEPA для текста на небольшом двунаправленном Transformer, сравнить базовое обучение с дополнительным SIGReg и проверить геометрию латентных представлений.

- **Данные:** Hugging Face `Salesforce/wikitext`, конфигурация `wikitext-2-raw-v1`, официальные раздельные train/validation/test.
- **Модель:** student context encoder (4 слоя, d=256), teacher с EMA, отдельный 2-слойный predictor.
- **Маска:** один непрерывный span длиной 15–35% последовательности.
- **Цель:** предсказать teacher-эмбеддинги **скрытых позиций**; teacher видит всю последовательность.
- **Контроль:** validation prediction loss, дисперсия, effective rank, masked-position retrieval, визуализация cosine similarity.
- **Эксперимент:** сравнивайте `sigreg_weight=0` и `sigreg_weight>0` при одинаковом seed.

**Важные ограничения:** это исследовательский прототип, не готовая языковая модель и не декодер пропущенных слов. Teacher-цели имеют доступ к полному контексту. Retrieval среди латентов не эквивалентен восстановлению токенов или измерению семантического понимания. Для доказательства преимуществ нужны контрольные модели MLM и downstream-бенчмарки.

**Запуск:** сверху вниз в Jupyter/Colab/Kaggle/DGX Spark. Первый запуск требует интернета для датасета и токенизатора. На DGX Spark не заменяйте уже рабочую сборку CUDA PyTorch обычным `pip install torch`.

In [ ]:
# Зависимости (выполнить один раз при необходимости):
# %pip install -q "transformers>=4.45,<6" "datasets>=3,<5" matplotlib tqdm scikit-learn
import copy, math, os, random, time
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm

print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 1. Конфигурация

Базовый эксперимент работает с 128 токенами, 4 слоями энкодера и 2 слоями предиктора. Для быстрой проверки установите `max_steps=30`; для основного прогона увеличьте число шагов и размер выборки.

**SIGReg** здесь реализован как дифференцируемое приближение теста Эппса–Палли: случайные одномерные проекции, сравнение эмпирической характеристической функции с `exp(-t²/2)`. Это исследовательская реализация принципа SIGReg, а не побитовая копия конкретного исследовательского репозитория.

In [ ]:
@dataclass
class Config:
    seed: int = 42
    dataset_name: str = "Salesforce/wikitext"
    dataset_config: str = "wikitext-2-raw-v1"  # для масштабирования: wikitext-103-raw-v1
    tokenizer_name: str = "gpt2"
    seq_len: int = 128
    d_model: int = 256
    heads: int = 4
    encoder_layers: int = 4
    predictor_layers: int = 2
    dropout: float = 0.1
    batch_size: int = 32
    max_train_sequences: int = 6000
    max_val_sequences: int = 600
    min_mask_frac: float = 0.15
    max_mask_frac: float = 0.35
    lr: float = 3e-4
    weight_decay: float = 0.05
    max_steps: int = 800
    ema_start: float = 0.996
    ema_end: float = 0.9995
    sigreg_weight: float = 0.0  # например 0.05 для варианта EMA + SIGReg
    sigreg_directions: int = 64
    sigreg_samples: int = 1024
    grad_clip: float = 1.0
    out_dir: str = "runs/text_i_jepa"
cfg = Config()
assert 0 < cfg.min_mask_frac <= cfg.max_mask_frac < 1
assert cfg.seq_len >= 16
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(cfg.seed)
print(cfg)

## 2. Данные и токенизация

Токенизатор GPT-2 применяется **только** для разбиения текста на токены: предварительно обученные веса GPT-2 в модель не загружаются. Это удобнее и позволяет исследовать именно обучение энкодера **с нуля**. Последовательности берутся из официальных, независимых split. Никакого случайного train/val деления после конкатенации не происходит.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_name)
raw = load_dataset(cfg.dataset_name, cfg.dataset_config)
print(raw)
class TokenBlocks(Dataset):
    def __init__(self, split, max_examples):
        lines = [str(s) for s in raw[split]["text"] if isinstance(s,str) and s.strip()]
        # Разделяем строки явным символом переноса.
        ids = tokenizer("\n".join(lines), add_special_tokens=False)["input_ids"]
        count = min(len(ids)//cfg.seq_len, max_examples)
        self.blocks = torch.tensor(ids[:count*cfg.seq_len],dtype=torch.long).view(count,cfg.seq_len)
        assert count > 0, "Недостаточно токенов в split"
        print(split, "tokens:",len(ids),"sequences:", count)
    def __len__(self): return len(self.blocks)
    def __getitem__(self,idx): return self.blocks[idx]
train_set = TokenBlocks("train",cfg.max_train_sequences)
val_set = TokenBlocks("validation",cfg.max_val_sequences)
train_loader = DataLoader(train_set,batch_size=cfg.batch_size,shuffle=True,drop_last=True,pin_memory=DEVICE.type=="cuda")
val_loader = DataLoader(val_set,batch_size=cfg.batch_size,shuffle=False,drop_last=False)
print("Example:",tokenizer.decode(train_set[0][:80]))

## 3. Формирование span-масок

Для каждого примера выбирается один непрерывный скрытый блок. Число скрытых токенов одинаково внутри батча, но начало блока различается. Student получает только видимые ID и **их исходные позиции**, teacher — полную последовательность; predictor получает видимые представления student и learnable mask-token на скрытых позициях.

В отличие от простого BERT `[MASK]`, скрытые токены **вообще не проходят через student encoder**.

In [ ]:
def sample_masks(batch_size, seq_len, min_frac, max_frac, device, deterministic=False):
    # Одинаковая длина span в пределах батча; для validation можно фиксировать seed отдельно.
    if deterministic:
        n_mask = round(seq_len*(min_frac+max_frac)/2)
        starts = (torch.arange(batch_size,device=device)*17)%(seq_len-n_mask+1)
    else:
        n_mask = random.randint(max(1,round(seq_len*min_frac)), min(seq_len-1,round(seq_len*max_frac)))
        starts = torch.randint(0,seq_len-n_mask+1,(batch_size,),device=device)
    positions = torch.arange(seq_len,device=device)[None,:].expand(batch_size,-1)
    hidden = (positions>=starts[:,None]) & (positions<starts[:,None]+n_mask)
    visible_pos = positions[~hidden].view(batch_size,seq_len-n_mask)
    target_pos = positions[hidden].view(batch_size,n_mask)
    return visible_pos,target_pos,hidden
sample = train_set[0]
v,t,h=sample_masks(1,cfg.seq_len,cfg.min_mask_frac,cfg.max_mask_frac,DEVICE,deterministic=True)
printed = ["[HIDDEN]" if bool(h[0,j]) else tokenizer.decode([int(sample[j])]) for j in range(cfg.seq_len)]
print("".join(printed))
print("Visible:",v.shape,"Target:",t.shape)

## 4. Архитектура

Student/Teacher — одинаковые двунаправленные Transformer Encoder. Predictor восстанавливает **позиции представлений**, а не текстовые ID. Разные позиции задаются обучаемыми positional embeddings.

Параметры teacher инициализируются копированием student; никакой gradient через teacher нет.

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self,c,vocab_size):
        super().__init__()
        self.tok=nn.Embedding(vocab_size,c.d_model)
        self.pos=nn.Embedding(c.seq_len,c.d_model)
        block=nn.TransformerEncoderLayer(d_model=c.d_model,nhead=c.heads,dim_feedforward=4*c.d_model,dropout=c.dropout,activation="gelu",batch_first=True,norm_first=True)
        self.net=nn.TransformerEncoder(block,c.encoder_layers,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(c.d_model)
    def forward(self,ids,positions):
        return self.norm(self.net(self.tok(ids)+self.pos(positions)))

class SpanPredictor(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.mask_token=nn.Parameter(torch.zeros(c.d_model))
        self.position=nn.Embedding(c.seq_len,c.d_model)
        block=nn.TransformerEncoderLayer(d_model=c.d_model,nhead=c.heads,dim_feedforward=c.d_model*4,dropout=c.dropout,activation="gelu",batch_first=True,norm_first=True)
        self.net=nn.TransformerEncoder(block,c.predictor_layers,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(c.d_model)
        nn.init.normal_(self.mask_token,std=0.02)
    def forward(self,visible_features,visible_positions,target_positions,seq_len):
        b=visible_features.shape[0]
        h=self.mask_token[None,None,:].expand(b,seq_len,-1).clone()
        h.scatter_(1,visible_positions[:,:,None].expand_as(visible_features),visible_features)
        all_pos=torch.arange(seq_len,device=h.device)[None,:].expand(b,-1)
        pred=self.norm(self.net(h+self.position(all_pos)))
        return pred.gather(1,target_positions[:,:,None].expand(-1,-1,pred.size(-1)))

student=TextEncoder(cfg,len(tokenizer)).to(DEVICE)
teacher=copy.deepcopy(student).to(DEVICE).eval()
for p in teacher.parameters(): p.requires_grad_(False)
predictor=SpanPredictor(cfg).to(DEVICE)
print("student",sum(p.numel() for p in student.parameters())/1e6,"M params")
print("predictor",sum(p.numel() for p in predictor.parameters())/1e6,"M params")

## 5. Потери, EMA и SIGReg

Основная ошибка: `MSE(predicted masked embeddings, teacher full-context embeddings)`.

EMA: `teacher <- m*teacher + (1-m)*student`, без градиента.

**Опционально SIGReg:** на наборе student-эмбеддингов берутся случайные направления `a_k`, вычисляются `u=z @ a_k` и сравниваются эмпирические характеристические функции `E exp(i*t*u)` со стандартным Gaussian `exp(-t²/2)`. Для сравнимости делаем SIGReg только на student-эмбеддингах; нормализация не должна уничтожать градиент. Статистики зависимых токенов внутри одного текста не являются независимыми наблюдениями, поэтому SIGReg здесь экспериментальный.

Обратите внимание: **EMA не гарантирует отсутствие коллапса**. Уровень качества латентов нужно измерять отдельно.

In [ ]:
def sigreg_loss(features, directions=64, samples=1024):
    # features: [B, V, D] ; используем небольшой поднабор для контроля памяти
    z=features.reshape(-1,features.shape[-1])
    if z.shape[0]>samples:
        idx=torch.randperm(z.shape[0],device=z.device)[:samples]
        z=z[idx]
    z=z.float()
    # Случайные проекции, нормированные на единичную длину.
    a=F.normalize(torch.randn(z.size(-1),directions,device=z.device),dim=0)
    # Масштаб sqrt(d) нужен, поскольку нормализованный a имеет координаты ~1/sqrt(d),
    # тогда unit-variance признаки дают unit-variance проекции.
    u=z@a
    t=torch.linspace(-3,3,21,device=z.device).view(1,1,-1)
    phase=u[:,:,None]*t
    empirical_real=phase.cos().mean(dim=0)
    empirical_imag=phase.sin().mean(dim=0)
    expected=torch.exp(-0.5*t.squeeze(0)**2)
    return ((empirical_real-expected)**2+empirical_imag**2).mean()

def ema_step(teacher,student,m):
    with torch.no_grad():
        for tp,sp in zip(teacher.parameters(),student.parameters()):
            tp.lerp_(sp,1-m)
        for tb,sb in zip(teacher.buffers(),student.buffers()):
            tb.copy_(sb)

def forward_batch(x, training=True):
    b,l=x.shape
    pos_v,pos_t,_=sample_masks(b,l,cfg.min_mask_frac,cfg.max_mask_frac,x.device,deterministic=not training)
    visible_ids=x.gather(1,pos_v)
    context=student(visible_ids,pos_v)
    pred=predictor(context,pos_v,pos_t,l)
    with torch.no_grad():
        all_pos=torch.arange(l,device=x.device)[None,:].expand(b,-1)
        target=teacher(x,all_pos).gather(1,pos_t[:,:,None].expand(-1,-1,cfg.d_model))
    # Масштаб MSE по нормализованным признакам уменьшает зависимость от нормы teacher
    regression=F.mse_loss(F.layer_norm(pred,(cfg.d_model,)),F.layer_norm(target,(cfg.d_model,)))
    regularizer=(sigreg_loss(context,cfg.sigreg_directions,cfg.sigreg_samples)
                 if cfg.sigreg_weight>0 else regression.new_zeros(()))
    return regression+cfg.sigreg_weight*regularizer,regression,regularizer,pred,target

optimizer=torch.optim.AdamW(list(student.parameters())+list(predictor.parameters()),lr=cfg.lr,weight_decay=cfg.weight_decay)
@torch.no_grad()
def validate(max_batches=10):
    student.eval();predictor.eval();teacher.eval()
    losses=[];ranks=[];stds=[];retrieval=[]
    for k,x in enumerate(val_loader):
        if k>=max_batches: break
        x=x.to(DEVICE)
        loss,pred_loss,reg,pred,target=forward_batch(x,False)
        losses.append(float(pred_loss))
        z=target.reshape(-1,cfg.d_model).float()
        stds.append(float(z.std(0).mean()))
        singular=torch.linalg.svdvals(z-z.mean(0))
        p=singular.square();p=p/p.sum().clamp_min(1e-9)
        ranks.append(float(torch.exp(-(p*(p+1e-12).log()).sum())))
        # retrieval по teacher-целям внутри батча: диагностическая, не смысловая accuracy
        pred_norm=F.normalize(pred.reshape(-1,cfg.d_model),dim=-1)
        target_norm=F.normalize(z,dim=-1)
        scores=pred_norm@target_norm.T
        retrieval.append(float((scores.argmax(-1)==torch.arange(len(scores),device=x.device)).float().mean()))
    return {"val_pred":float(np.mean(losses)),"target_std":float(np.mean(stds)),
            "effective_rank":float(np.mean(ranks)),"masked_retrieval_at1":float(np.mean(retrieval))}
print("Initial:",validate(3))

## 6. Обучение

Важные свойства: отдельный fixed validation split; клиппинг градиента; плавное обновление EMA; логирование train/val статистик; checkpoint. Чтобы сравнить `sigreg_weight=0` и `sigreg_weight=0.05`, перезапустите ноутбук с одинаковым seed и сохраните результаты в разных каталогах.

In [ ]:
history=[]
Path(cfg.out_dir).mkdir(parents=True,exist_ok=True)
student.train();predictor.train()
it=iter(train_loader)
start=time.time()
for step in tqdm(range(1,cfg.max_steps+1),desc="Text-I-JEPA"):
    try: x=next(it)
    except StopIteration:
        it=iter(train_loader);x=next(it)
    x=x.to(DEVICE,non_blocking=True)
    student.train();predictor.train()
    total,pred_loss,reg,_,_=forward_batch(x,True)
    optimizer.zero_grad(set_to_none=True)
    total.backward()
    grad_norm=nn.utils.clip_grad_norm_(list(student.parameters())+list(predictor.parameters()),cfg.grad_clip)
    optimizer.step()
    frac=(step-1)/max(1,cfg.max_steps-1)
    momentum=cfg.ema_start+(cfg.ema_end-cfg.ema_start)*frac
    ema_step(teacher,student,momentum)
    if step==1 or step%50==0 or step==cfg.max_steps:
        row={"step":step,"train_pred":float(pred_loss.detach()),"sigreg":float(reg.detach()),
             "grad_norm":float(grad_norm),"ema_m":momentum,"seconds":time.time()-start}
        row.update(validate(5))
        history.append(row)
        print(row)
torch.save({"student":student.state_dict(),"teacher":teacher.state_dict(),
            "predictor":predictor.state_dict(),"cfg":vars(cfg),"history":history},
           Path(cfg.out_dir)/"checkpoint.pt")
print("Saved:",Path(cfg.out_dir)/"checkpoint.pt")

## 7. Графики обучения и диагностика коллапса

Смотрите не только на loss: низкая ошибка предсказания **может сопровождаться коллапсом**. `target_std` около нуля или крайне низкий `effective_rank` при большой размерности — тревожные признаки. Retrieval@1 сравнивает предсказания с позиционными teacher-целями; это лишь вспомогательная диагностическая задача.

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(15,4))
for ax,key in zip(axs,["train_pred","val_pred","effective_rank"]):
    ax.plot([h["step"] for h in history],[h[key] for h in history],marker="o",markersize=3)
    ax.set_title(key);ax.set_xlabel("optimizer step");ax.grid(alpha=.25)
plt.tight_layout();plt.show()
print("Latest:",history[-1] if history else "Run training cell first")

## 8. Визуализация предсказанных латентов и скрытого блока

Отдельно покажем текст с пропуском и матрицу косинусных сходств между предсказанными и целевыми скрытыми позициями. **Это не генерация слов**: успешный латентный прогноз не позволяет автоматически вывести исходные текстовые токены.

In [ ]:
student.eval();predictor.eval();teacher.eval()
x=val_set[0][None,:].to(DEVICE)
with torch.no_grad():
    pos_v,pos_t,hidden=sample_masks(1,cfg.seq_len,cfg.min_mask_frac,cfg.max_mask_frac,DEVICE,deterministic=True)
    context=student(x.gather(1,pos_v),pos_v)
    pred=predictor(context,pos_v,pos_t,cfg.seq_len)
    all_pos=torch.arange(cfg.seq_len,device=DEVICE)[None,:]
    targets=teacher(x,all_pos).gather(1,pos_t[:,:,None].expand(-1,-1,cfg.d_model))
    sim=F.normalize(pred[0],dim=-1)@F.normalize(targets[0],dim=-1).T
masked_text="".join("[██]" if bool(hidden[0,i]) else tokenizer.decode([int(x[0,i])]) for i in range(cfg.seq_len))
print("CONTEXT WITH HIDDEN SPAN:\n",masked_text)
print("\nTRUE HIDDEN TEXT (shown only for inspection):\n",tokenizer.decode(x[0,pos_t[0]].tolist()))
plt.figure(figsize=(7,6));plt.imshow(sim.cpu(),vmin=-1,vmax=1,cmap="coolwarm")
plt.colorbar(label="cosine similarity");plt.xlabel("target hidden token index");plt.ylabel("predicted hidden token index")
plt.title("Text-I-JEPA: hidden-position latent similarity");plt.tight_layout();plt.show()

## 9. Что этот эксперимент доказывает и чего пока не доказывает

Успешный результат: модель предсказывает teacher-эмбеддинги скрытого текста лучше исходной сети, при этом сохраняет разнообразные и полноранговые представления на validation.

**Недостаточно** считать низкий loss доказательством понимания: позиционные признаки и локальные статистики могут давать хороший результат без глубокой семантики. Для строгого исследования нужны:
1. Контрольная MLM-модель при равном числе параметров/токенов/compute.
2. Linear probes на классификацию, entailment или семантические сходства.
3. Тест случайного замешивания целевых токенов и оценка устойчивости.
4. Абляция `sigreg_weight` при одинаковом seed и бюджете.
5. Увеличение датасета до WikiText-103 или других очищенных коллекций.

**Следующий шаг:** присоединить небольшой text decoder или contrastive text embedding head, чтобы явно оценить восстановление значения скрытых фрагментов. Не путать восстановление эмбеддинга с точным восстановлением токена.